In [1]:
import pandas as pd 
from kafka import KafkaConsumer, KafkaProducer
from time import sleep
from json import dumps
import json

In [2]:
producer = KafkaProducer(bootstrap_servers=['18.118.100.83:9093'],
                         value_serializer=lambda x: dumps(x).encode('utf-8'))

In [6]:
producer.send('demo_testing2', value={'name': 'krishna', 'age': 24})

In [3]:
import json
import logging
import os
import random
import socket
import time
import uuid
from datetime import datetime, timezone

from confluent_kafka import KafkaError, KafkaException, Producer
from dotenv import load_dotenv


load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger(__name__)


KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS")
KAFKA_TEST_TOPIC = os.getenv("KAFKA_TEST_TOPIC", "stock-trades-test")

SYMBOLS = ["AAPL", "MSFT", "NVDA", "TSLA", "AMZN"]

BASE_PRICES = {
    "AAPL": 200.00,
    "MSFT": 450.00,
    "NVDA": 140.00,
    "TSLA": 300.00,
    "AMZN": 220.00,
}


def validate_configuration() -> None:
    """Validate required environment variables before starting."""

    if not KAFKA_BOOTSTRAP_SERVERS:
        raise ValueError(
            "KAFKA_BOOTSTRAP_SERVERS is missing. "
            "Add it to the .env file."
        )


def build_producer() -> Producer:
    """Create and configure the Kafka producer."""

    producer_config = {
        "bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS,
        "client.id": f"local-test-producer-{socket.gethostname()}",

        # Wait for Kafka to acknowledge the message.
        "acks": "all",

        # Helps prevent duplicate messages caused by producer retries.
        "enable.idempotence": True,

        # Allow Kafka to briefly collect records into more efficient batches.
        "linger.ms": 50,

        # Maximum time Kafka may spend trying to deliver a message.
        "delivery.timeout.ms": 120000,

        # Timeout for individual broker requests.
        "request.timeout.ms": 30000,
    }

    return Producer(producer_config)


def create_test_trade() -> dict:
    """Generate a simulated stock-trade event."""

    symbol = random.choice(SYMBOLS)

    price_change = random.uniform(-2.50, 2.50)
    simulated_price = round(BASE_PRICES[symbol] + price_change, 2)

    event_time = datetime.now(timezone.utc)
    event_timestamp_ms = int(event_time.timestamp() * 1000)

    return {
        "schema_version": "1.0",
        "event_id": str(uuid.uuid4()),
        "event_type": "trade",
        "source": "simulator",
        "symbol": symbol,
        "price": simulated_price,
        "volume": random.randint(1, 1000),
        "trade_conditions": [],
        "event_timestamp_ms": event_timestamp_ms,
        "event_timestamp_utc": event_time.isoformat(),
        "ingestion_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }


def delivery_report(error, message) -> None:
    """
    Called by the Kafka client when a message is either delivered
    successfully or permanently fails.
    """

    if error is not None:
        logger.error(
            "Message delivery failed | error=%s",
            error,
        )
        return

    logger.info(
        "Delivered | topic=%s | partition=%s | offset=%s | key=%s",
        message.topic(),
        message.partition(),
        message.offset(),
        message.key().decode("utf-8") if message.key() else None,
    )


def send_test_messages(
    producer: Producer,
    number_of_messages: int = 25,
    delay_seconds: float = 1.0,
) -> None:
    """Generate and send simulated trade messages."""

    logger.info(
        "Starting producer | broker=%s | topic=%s",
        KAFKA_BOOTSTRAP_SERVERS,
        KAFKA_TEST_TOPIC,
    )

    for message_number in range(1, number_of_messages + 1):
        trade_event = create_test_trade()

        message_key = trade_event["symbol"]
        message_value = json.dumps(trade_event)

        delivered = False

        while not delivered:
            try:
                producer.produce(
                    topic=KAFKA_TEST_TOPIC,
                    key=message_key.encode("utf-8"),
                    value=message_value.encode("utf-8"),
                    callback=delivery_report,
                )

                delivered = True

            except BufferError:
                logger.warning(
                    "Producer queue is full. Waiting for pending deliveries."
                )

                producer.poll(1)

            except KafkaException as exc:
                logger.exception(
                    "Kafka producer error while sending message %s",
                    message_number,
                )
                raise exc

        # Trigger delivery callbacks for completed Kafka requests.
        producer.poll(0)

        logger.info(
            "Queued message %s/%s | symbol=%s | price=%s | volume=%s",
            message_number,
            number_of_messages,
            trade_event["symbol"],
            trade_event["price"],
            trade_event["volume"],
        )

        time.sleep(delay_seconds)

    logger.info("Waiting for outstanding Kafka messages to be delivered.")

    undelivered_messages = producer.flush(timeout=15)

    if undelivered_messages == 0:
        logger.info("All messages were delivered successfully.")
    else:
        raise KafkaException(
            KafkaError(
                KafkaError._MSG_TIMED_OUT,
                f"{undelivered_messages} message(s) were not delivered.",
            )
        )


def main() -> None:
    validate_configuration()

    producer = build_producer()

    send_test_messages(
        producer=producer,
        number_of_messages=25,
        delay_seconds=1,
    )


if __name__ == "__main__":
    try:
        main()

    except KeyboardInterrupt:
        logger.warning("Producer stopped by user.")

    except Exception:
        logger.exception("Producer terminated because of an error.")
        raise

2026-08-02 19:12:55,457 | INFO | Starting producer | broker=3.95.201.137:9093 | topic=stock-trades-test
2026-08-02 19:12:55,458 | INFO | Queued message 1/25 | symbol=NVDA | price=138.91 | volume=952
2026-08-02 19:12:56,461 | INFO | Queued message 2/25 | symbol=NVDA | price=138.15 | volume=347
2026-08-02 19:12:57,467 | INFO | Delivered | topic=stock-trades-test | partition=1 | offset=0 | key=NVDA
2026-08-02 19:12:57,469 | INFO | Delivered | topic=stock-trades-test | partition=1 | offset=1 | key=NVDA
2026-08-02 19:12:57,470 | INFO | Queued message 3/25 | symbol=AAPL | price=199.48 | volume=157
2026-08-02 19:12:58,481 | INFO | Delivered | topic=stock-trades-test | partition=0 | offset=0 | key=AAPL
2026-08-02 19:12:58,483 | INFO | Queued message 4/25 | symbol=AMZN | price=221.21 | volume=24
2026-08-02 19:12:59,497 | INFO | Delivered | topic=stock-trades-test | partition=1 | offset=2 | key=AMZN
2026-08-02 19:12:59,499 | INFO | Queued message 5/25 | symbol=AMZN | price=217.95 | volume=713
20

In [1]:
import json
import logging
import os
import random
import socket
import time
import uuid
from datetime import datetime, timezone

from confluent_kafka import Producer
from dotenv import load_dotenv


load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger(__name__)


KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS")
KAFKA_TOPIC = os.getenv(
    "KAFKA_TEST_TOPIC",
    "stock-trades-test",
)

SYMBOLS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "TSLA",
    "AMZN",
]

# Starting prices only.
# Prices will move gradually instead of being completely random.
CURRENT_PRICES = {
    "AAPL": 220.00,
    "MSFT": 490.00,
    "NVDA": 180.00,
    "TSLA": 320.00,
    "AMZN": 230.00,
}


def build_producer():
    return Producer(
        {
            "bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS,
            "client.id": f"stock-simulator-{socket.gethostname()}",
            "acks": "all",
            "enable.idempotence": True,
            "linger.ms": 50,
        }
    )


producer = build_producer()


def delivery_report(error, message):

    if error:
        logger.error(
            "Delivery failed | %s",
            error,
        )
        return

    logger.info(
        "Delivered | symbol=%s | partition=%s | offset=%s",
        message.key().decode("utf-8"),
        message.partition(),
        message.offset(),
    )


def create_simulated_trade():

    symbol = random.choice(SYMBOLS)

    # Simulate a small market price movement.
    price_change_pct = random.uniform(
        -0.0015,
        0.0015,
    )

    CURRENT_PRICES[symbol] *= (
        1 + price_change_pct
    )

    price = round(
        CURRENT_PRICES[symbol],
        2,
    )

    # Common trade sizes.
    volume = random.choice(
        [
            10,
            20,
            40,
            50,
            80,
            100,
            200,
            500,
            1000,
        ]
    )

    event_time = datetime.now(
        timezone.utc
    )

    event_timestamp_ms = int(
        event_time.timestamp() * 1000
    )

    return {
        "schema_version": "1.0",
        "event_id": str(uuid.uuid4()),
        "event_type": "trade",
        "source": "simulator",
        "symbol": symbol,
        "price": price,
        "volume": float(volume),
        "trade_conditions": ["SIMULATED"],
        "event_timestamp_ms": event_timestamp_ms,
        "event_timestamp_utc": event_time.isoformat(),
        "ingestion_timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    }


def publish_trade(trade):

    producer.produce(
        topic=KAFKA_TOPIC,
        key=trade["symbol"].encode("utf-8"),
        value=json.dumps(trade).encode("utf-8"),
        callback=delivery_report,
    )

    producer.poll(0)


def main():

    logger.info(
        "Starting simulated market stream | "
        "Kafka=%s | topic=%s",
        KAFKA_BOOTSTRAP_SERVERS,
        KAFKA_TOPIC,
    )

    message_count = 0

    try:

        while True:

            trade = create_simulated_trade()

            publish_trade(trade)

            message_count += 1

            logger.info(
                "Simulated trade #%s | "
                "symbol=%s | price=%s | volume=%s",
                message_count,
                trade["symbol"],
                trade["price"],
                trade["volume"],
            )

            # Random delay makes the stream less artificial.
            time.sleep(
                random.uniform(
                    0.15,
                    0.75,
                )
            )

    except KeyboardInterrupt:

        logger.info(
            "Stopping simulated market stream."
        )

    finally:

        remaining = producer.flush(
            timeout=15
        )

        logger.info(
            "Producer closed | undelivered=%s",
            remaining,
        )


if __name__ == "__main__":
    main()

2026-08-08 22:32:35,784 | INFO | Starting simulated market stream | Kafka=54.172.250.154:9093 | topic=stock-trades-test
2026-08-08 22:32:35,787 | INFO | Simulated trade #1 | symbol=TSLA | price=320.29 | volume=50.0
2026-08-08 22:32:36,469 | INFO | Simulated trade #2 | symbol=AMZN | price=230.08 | volume=50.0
2026-08-08 22:32:36,925 | INFO | Simulated trade #3 | symbol=NVDA | price=180.19 | volume=200.0
2026-08-08 22:32:37,191 | INFO | Delivered | symbol=TSLA | partition=0 | offset=0
2026-08-08 22:32:37,193 | INFO | Delivered | symbol=AMZN | partition=1 | offset=0
2026-08-08 22:32:37,194 | INFO | Delivered | symbol=NVDA | partition=1 | offset=1
2026-08-08 22:32:37,195 | INFO | Simulated trade #4 | symbol=AAPL | price=219.95 | volume=200.0
2026-08-08 22:32:37,535 | INFO | Delivered | symbol=AAPL | partition=0 | offset=1
2026-08-08 22:32:37,537 | INFO | Simulated trade #5 | symbol=AMZN | price=229.88 | volume=20.0
2026-08-08 22:32:37,901 | INFO | Delivered | symbol=AMZN | partition=1 | of

In [2]:
import json
import os
import uuid
from datetime import datetime, timezone

from confluent_kafka import Producer
from dotenv import load_dotenv


load_dotenv()

KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS")
KAFKA_TOPIC = os.getenv("KAFKA_TEST_TOPIC", "stock-trades-test")


producer = Producer({
    "bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS,
    "acks": "all",
    "enable.idempotence": True,
})


def publish(message_key, record):

    producer.produce(
        topic=KAFKA_TOPIC,
        key=message_key.encode("utf-8"),
        value=json.dumps(record).encode("utf-8"),
    )

    producer.poll(0)


def make_base_trade(symbol="AAPL"):

    now = datetime.now(timezone.utc)

    return {
        "schema_version": "1.0",
        "event_id": str(uuid.uuid4()),
        "event_type": "trade",
        "source": "simulator",
        "symbol": symbol,
        "price": 220.50,
        "volume": 100.0,
        "trade_conditions": ["QUARANTINE_TEST"],
        "event_timestamp_ms": int(now.timestamp() * 1000),
        "event_timestamp_utc": now.isoformat(),
        "ingestion_timestamp_utc": now.isoformat(),
    }


# ---------------------------------------------------------
# 1. Missing event_id
# ---------------------------------------------------------

trade_1 = make_base_trade("AAPL")
trade_1.pop("event_id")

publish(
    "AAPL",
    trade_1,
)


# ---------------------------------------------------------
# 2. Unsupported symbol
# ---------------------------------------------------------

trade_2 = make_base_trade("META")

publish(
    "META",
    trade_2,
)


# ---------------------------------------------------------
# 3. Invalid negative price
# ---------------------------------------------------------

trade_3 = make_base_trade("MSFT")
trade_3["price"] = -100.0

publish(
    "MSFT",
    trade_3,
)


# ---------------------------------------------------------
# 4. Kafka key does not match symbol
# ---------------------------------------------------------

trade_4 = make_base_trade("NVDA")

publish(
    "TSLA",
    trade_4,
)


producer.flush()

print("Published 4 intentionally invalid trades.")

Published 4 intentionally invalid trades.
